<a href="https://colab.research.google.com/github/Text-Machine/mask-predict/blob/main/compute_influence_improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

In [ ]:
!git clone https://github.com/Text-Machine/mask-predict.git

In [ ]:
%cd mask-predict

In [ ]:
!pip install -e .

In [1]:
#import ast
#import matplotlib.pyplot as plt
#import seaborn as sns
#import numpy as np
#import pandas as pd
#import torch
#from transformers import AutoTokenizer, AutoModelForMaskedLM
#from captum.attr import IntegratedGradients
from explain import *
import pandas as pd

In [2]:
collection = 'hmd'
maskedToken = 'machine'
clusterName = 'slave'

In [ ]:
#!unzip -o "{collection}_{maskedToken}_clusters.tsv.zip"

In [3]:
data_df = pd.read_csv(f"masking_data/{collection}_{maskedToken}_clusters.tsv", sep="\t")
data_df = data_df.sort_values(by=f"{clusterName}_1760_1900", ascending=False).reset_index(drop=True)

data_df.head()

,Unnamed: 0,item_code,issue_code,publication_code,prevSentence,currentSentence,markedSentence,maskedSentence,nextSentence,targetExpression,...,girl_1760_1900,slave_1760_1900,artisan_1760_1900,woman_1760_1900,machine_contemporary,boy_contemporary,girl_contemporary,slave_contemporary,artisan_contemporary,woman_contemporary
0,8773,art0058,18540204,2645,He thinks the happiness and respectability of ...,Technical skill and exact work • mauship degra...,Technical skill and exact work • mauship degra...,Technical skill and exact work • mauship degra...,"It is not work that is objected to , but the n...",machine,...,0.287,0.934,0.357,0.355,0.114,0.164,0.146,0.270,0.183,0.173
1,26694,art0114,18650401,2194,The Secretary of War at Richmond has also auth...,Without this half the advantage of the measure...,Without this half the advantage of the measure...,Without this half the advantage of the measure...,The military news is still conflicting and unc...,machines,...,0.301,0.797,0.374,0.360,0.176,0.201,0.214,0.322,0.222,0.249
2,33572,art0016,18650403,2642,If we chance to turn back to the writings of t...,"Our fields seemed blasted, and no longer able ...","Our fields seemed blasted , and no longer able...","Our fields seemed blasted , and no longer able...",It was a time when the phrase Two Nations was ...,machines,...,0.324,0.565,0.337,0.369,0.188,0.260,0.260,0.467,0.278,0.299
3,26405,art0010,18641001,2194,Instead of each elector feeling elevated by th...,The act of receiving a bribe is of itself des ...,The act of receiving a bribe is of itself des ...,The act of receiving a bribe is of itself des ...,These and other similar reasons have so disgus...,machine,...,0.257,0.563,0.283,0.298,0.142,0.190,0.194,0.268,0.143,0.205
4,26404,art0098,18641001,2194,Instead of ech elector feeling elevated by the...,The act of receiving a bribe is of itself de •...,The act of receiving a bribe is of itself de •...,The act of receiving a bribe is of itself de •...,These and other similar reasons have so disgus...,machine,...,0.269,0.558,0.293,0.309,0.123,0.177,0.180,0.256,0.136,0.195


In [10]:
# def pick_device():
#     if torch.backends.mps.is_available():
#         return "mps"
#     if torch.cuda.is_available():
#         return "cuda"
#     return "cpu"

# def parse_pred_column(cell, top_n=5):
#     # expected format: "[('word1', score1), ('word2', score2), ...]"
#     items = ast.literal_eval(cell)
#     return [w for w, _ in items[:top_n]]

# def build_texts_targets(df, start, end, pred_col, top_n=5):
#     subset = df.iloc[start:end].copy()
#     texts = subset["maskedSentence"].tolist()
#     targets = [parse_pred_column(x, top_n=top_n) for x in subset[pred_col].tolist()]
#     return texts, targets

In [11]:
# class MaskedLMExplainer:
#     def __init__(self, model_name="bert-base-uncased", device=None):
#         self.device = device or pick_device()
#         self.tokenizer = AutoTokenizer.from_pretrained(model_name)
#         self.model = AutoModelForMaskedLM.from_pretrained(model_name).to(self.device)
#         self.model.eval()
#         self.ig = IntegratedGradients(self.forward_func)

#     # Supports K mask positions / K target tokens per example
#     def forward_func(self, inputs_embeds, attention_mask, mask_indices, target_indices, valid_positions):
#         logits = self.model(inputs_embeds=inputs_embeds, attention_mask=attention_mask).logits  # (B, L, V)
#         bsz = logits.size(0)

#         # (B, K, V): logits at each masked position
#         mask_logits = logits[
#             torch.arange(bsz, device=logits.device).unsqueeze(1),
#             mask_indices
#         ]
#         log_probs = torch.log_softmax(mask_logits, dim=-1)  # (B, K, V)

#         # (B, K): log p(target_token_k | context)
#         target_lp = log_probs.gather(-1, target_indices.unsqueeze(-1)).squeeze(-1)

#         # Sum only valid positions
#         return (target_lp * valid_positions).sum(dim=1)  # (B,)

#     def _target_to_token_ids(self, text_target):
#         ids = self.tokenizer(text_target, add_special_tokens=False)["input_ids"]
#         return ids

#     def _expand_single_mask(self, text, n_masks):
#         mask = self.tokenizer.mask_token
#         if text.count(mask) != 1:
#             raise ValueError("Input sentence must contain exactly one [MASK] before expansion.")
#         return text.replace(mask, " ".join([mask] * n_masks), 1)
    
#     def _aggregate_tokens_to_words(self, token_rows, agg="mean"):
#         """
#         token_rows: list[(token, score)]
#         Returns: list[(word, aggregated_score)]
#         """
#         if agg not in {"mean", "max"}:
#             raise ValueError("agg must be 'mean' or 'max'")

#         def reduce_scores(scores):
#             return float(sum(scores) / len(scores)) if agg == "mean" else float(max(scores))

#         word_rows = []
#         current_word = None
#         current_scores = []

#         for tok, score in token_rows:
#             # BERT WordPiece continuation
#             if tok.startswith("##"):
#                 piece = tok[2:]
#                 if current_word is None:
#                     current_word = piece
#                     current_scores = [score]
#                 else:
#                     current_word += piece
#                     current_scores.append(score)
#             else:
#                 if current_word is not None:
#                     word_rows.append((current_word, reduce_scores(current_scores)))
#                 current_word = tok
#                 current_scores = [score]

#         if current_word is not None:
#             word_rows.append((current_word, reduce_scores(current_scores)))

#         return word_rows

#     def explain(
#         self,
#         texts,
#         target_words_list,
#         normalize=True,
#         drop_special=True,
#         return_word_scores=True,
#         word_agg="mean",  # "mean" or "max"
#     ):
#         if len(texts) != len(target_words_list):
#             raise ValueError("texts and target_words_list must have same length")

#         all_results = []
#         for text, targets in zip(texts, target_words_list):
#             sent_out = {}

#             for target in targets:
#                 target_ids = self._target_to_token_ids(target)
#                 if len(target_ids) == 0:
#                     sent_out[target] = {"skipped": True, "reason": "empty tokenization"}
#                     continue

#                 text_k = self._expand_single_mask(text, len(target_ids))
#                 enc = self.tokenizer([text_k], return_tensors="pt", padding=True, truncation=True)
#                 input_ids = enc["input_ids"].to(self.device)
#                 attention_mask = enc["attention_mask"].to(self.device)

#                 mask_pos = (input_ids[0] == self.tokenizer.mask_token_id).nonzero(as_tuple=False).flatten()
#                 if mask_pos.numel() != len(target_ids):
#                     sent_out[target] = {
#                         "skipped": True,
#                         "reason": f"mask count ({mask_pos.numel()}) != target token count ({len(target_ids)})"
#                     }
#                     continue

#                 emb = self.model.get_input_embeddings()(input_ids)
#                 baseline_ids = torch.full_like(input_ids, self.tokenizer.pad_token_id)
#                 baseline_ids[0, mask_pos] = self.tokenizer.mask_token_id
#                 base = self.model.get_input_embeddings()(baseline_ids)

#                 mpos = mask_pos.unsqueeze(0)
#                 tid = torch.tensor(target_ids, device=self.device).unsqueeze(0)
#                 valid = torch.ones_like(tid, dtype=torch.float32, device=self.device)

#                 attrs = self.ig.attribute(
#                     inputs=emb,
#                     baselines=base,
#                     additional_forward_args=(attention_mask, mpos, tid, valid),
#                 ).sum(dim=-1).squeeze(0)

#                 if normalize:
#                     attrs = attrs / attrs.norm().clamp_min(1e-12)

#                 attrs[mask_pos] = 0.0

#                 tokens = self.tokenizer.convert_ids_to_tokens(input_ids[0].tolist())
#                 token_rows = []
#                 for tok, val, tid_ in zip(tokens, attrs.tolist(), input_ids[0].tolist()):
#                     if drop_special and tid_ in {
#                         self.tokenizer.cls_token_id,
#                         self.tokenizer.sep_token_id,
#                         self.tokenizer.pad_token_id,
#                     }:
#                         continue
#                     token_rows.append((tok, float(val)))

#                 result_obj = {
#                     "skipped": False,
#                     "target_token_ids": target_ids,
#                     "token_attributions": token_rows,
#                 }

#                 if return_word_scores:
#                     result_obj["word_attributions"] = self._aggregate_tokens_to_words(
#                         token_rows, agg=word_agg
#                     )

#                 sent_out[target] = result_obj

#             all_results.append(sent_out)

#         return all_results


In [6]:
iloc_range = (0, 30)  # Adjust as needed
texts, targets = build_texts_targets(
    data_df, start=iloc_range[0], end=iloc_range[1],
    pred_col="pred_bert_1760_1900", top_n=5
)


In [5]:

explainer = MaskedLMExplainer(model_name="Livingwithmachines/bert_1760_1900", device=pick_device())
results = explainer.explain(texts, targets, word_agg="max")


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: Livingwithmachines/bert_1760_1900
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:

df = result_as_dataframe(results, ['slave', 'machine'])
df[df['Target'] == 'slave'].groupby('Token').agg(avg=('Score','mean'), count=('Score', 'count')).sort_values(by='avg', ascending=False)
                                                                                                             


,avg,count
Token,,
field,0.540928,1
tyrant,0.513091,1
workers,0.482052,2
slave,0.475154,1
bribe,0.457101,2
...,...,...
p,-0.009398,1
receiving,-0.009853,2
same,-0.014579,1


In [ ]:
# def compare_explainers(explainer_1, explainer_2, texts, targets, level="word", word_agg="mean"):
#     r1 = explainer_1.explain(texts, targets, return_word_scores=(level == "word"), word_agg=word_agg)
#     r2 = explainer_2.explain(texts, targets, return_word_scores=(level == "word"), word_agg=word_agg)

#     key = "word_attributions" if level == "word" else "token_attributions"

#     comparisons = []
#     for i in range(len(texts)):
#         out = {}
#         for target in targets[i]:
#             a = r1[i][target]
#             b = r2[i][target]
#             if a.get("skipped") or b.get("skipped"):
#                 out[target] = {"skipped": True, "model1": a, "model2": b}
#                 continue

#             rows1 = a[key]
#             rows2 = b[key]

#             t1 = [x[0] for x in rows1]
#             t2 = [x[0] for x in rows2]
#             if t1 != t2:
#                 raise ValueError(f"{level.capitalize()} mismatch at sentence {i}, target '{target}'")

#             s1 = [x[1] for x in rows1]
#             s2 = [x[1] for x in rows2]
#             out[target] = list(zip(t1, s1, s2, [x - y for x, y in zip(s1, s2)]))
#         comparisons.append(out)
#     return comparisons


In [8]:

explainer_new = MaskedLMExplainer("Livingwithmachines/bert_1760_1900", device=pick_device())
explainer_old = MaskedLMExplainer("Livingwithmachines/bert_1760_1850", device=pick_device())

# Correct shape: one target-list per sentence
single_target = [["slave"] for _ in texts]

comparison = compare_explainers(explainer_old,explainer_new, texts, single_target, level="word", word_agg="max")

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: Livingwithmachines/bert_1760_1900
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: Livingwithmachines/bert_1760_1850
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:


# def summarize_top_predictors(results, target, top_n=10):
#     """
#     Aggregate attributions across all sentences for a single target word.
    
#     Args:
#         results: output from explainer.explain()
#         target: target word to summarize
#         top_n: number of top predictors to return
    
#     Returns:
#         list of (word, mean_attribution, std_attribution, count)
#     """
#     word_scores = {}
    
#     for sent_result in results:
#         if target not in sent_result or sent_result[target].get("skipped"):
#             continue
        
#         word_attrs = sent_result[target]["word_attributions"]
#         for word, score in word_attrs:
#             if word not in word_scores:
#                 word_scores[word] = []
#             word_scores[word].append(score)
    
#     # Compute statistics
#     stats = []
#     for word, scores in word_scores.items():
#         mean_score = float(np.mean(scores))
#         std_score = float(np.std(scores))
#         count = len(scores)
#         stats.append((word, mean_score, std_score, count))
    
#     # Sort by mean attribution (descending)
#     stats.sort(key=lambda x: abs(x[1]), reverse=True)
    
#     return stats[:top_n]

# Example usage
top_predictors = summarize_top_predictors(results, target="slave", top_n=15)
print(f"Top predictors for 'slave':\n")
print(f"{'Word':>15} {'Mean':>10} {'Std':>10} {'Count':>8}")
print("-" * 45)
for word, mean, std, count in top_predictors:
    print(f"{word:>15} {mean:>10.4f} {std:>10.4f} {count:>8}")

Top predictors for 'slave':

           Word       Mean        Std    Count
---------------------------------------------
          field     0.5409     0.0000        1
         tyrant     0.5131     0.0000        1
        workers     0.4821     0.0000        2
          slave     0.4752     0.0000        1
          bribe     0.4571     0.0060        2
        federal     0.4500     0.0000        2
          beset     0.3796     0.0000        1
            man     0.3388     0.1315        2
           mere     0.3050     0.0000        1
          negro     0.2840     0.0000        1
         labour     0.2342     0.0000        1
        slavery     0.2324     0.0000        1
   confederates     0.2053     0.0000        2
         merely     0.2038     0.0000        1
        freedom     0.1994     0.0000        1


In [10]:
# Cell 10: Analyze model comparison
# ...existing code...
# def analyze_comparison(comparison, target, top_n=15):
#     """
#     Summarize differences between two models across all sentences.
#     Returns:
#         list of (word, model1_score, model2_score, difference, std_difference)
#     """
#     word_diffs = {}

#     for sent_comp in comparison:
#         # sent_comp should be dict: {target: rows_or_skipdict}
#         if not isinstance(sent_comp, dict) or target not in sent_comp:
#             continue

#         entry = sent_comp[target]

#         # Skip record format: {"skipped": True, ...}
#         if isinstance(entry, dict):
#             if entry.get("skipped", False):
#                 continue
#             # unknown dict shape -> skip safely
#             continue

#         # Normal record format: list[(word, s1, s2, diff)]
#         if not isinstance(entry, list):
#             continue

#         for word, s1, s2, diff in entry:
#             word_diffs.setdefault(word, []).append((s1, s2, diff))

#     stats = []
#     for word, deltas in word_diffs.items():
#         s1_vals = [x[0] for x in deltas]
#         s2_vals = [x[1] for x in deltas]
#         diff_vals = [x[2] for x in deltas]

#         mean_s1 = float(np.mean(s1_vals))
#         mean_s2 = float(np.mean(s2_vals))
#         mean_diff = float(np.mean(diff_vals))
#         std_diff = float(np.std(diff_vals))

#         stats.append((word, mean_s1, mean_s2, mean_diff, std_diff))

#     stats.sort(key=lambda x: abs(x[3]), reverse=True)
#     return stats[:top_n]


# Example usage
comparison_stats = analyze_comparison(comparison, target="slave", top_n=15)
print(f"Top diverging words between models for 'slave':\n")
print(f"{'Word':>15} {'Model1':>12} {'Model2':>12} {'Diff':>12} {'Std':>10}")
print("-" * 62)
for word, m1, m2, diff, std in comparison_stats:
    print(f"{word:>15} {m1:>12.4f} {m2:>12.4f} {diff:>12.4f} {std:>10.4f}")

Top diverging words between models for 'slave':

           Word       Model1       Model2         Diff        Std
--------------------------------------------------------------
      linchpins       0.4921       0.0981       0.3939     0.0000
        federal       0.0758       0.4500      -0.3742     0.0000
        workers       0.1161       0.4821      -0.3659     0.0000
         cotton       0.0610       0.4184      -0.3574     0.0000
          beset       0.0334       0.3796      -0.3462     0.0000
          field       0.0866       0.3741      -0.2875     0.1525
              )      -0.1412       0.1169      -0.2582     0.0000
          negro       0.0447       0.2840      -0.2393     0.0000
          slave       0.2494       0.4752      -0.2257     0.0000
     dispatched       0.0031       0.2276      -0.2245     0.0000
       brothers       0.1115       0.3247      -0.2132     0.0581
            man       0.1314       0.3388      -0.2074     0.1065
           lace      -0.0487  

In [ ]:
# ...existing code...
# from html import escape
# from IPython.display import HTML, display
# import uuid

# def _attr_to_rgba(score, max_abs):
#     if max_abs <= 0:
#         return "rgba(128,128,128,0.10)"
#     strength = min(abs(score) / max_abs, 1.0)
#     alpha = 0.10 + 0.75 * strength
#     if score >= 0:
#         return f"rgba(30, 136, 229, {alpha:.3f})"
#     return f"rgba(229, 57, 53, {alpha:.3f})"

# def highlight_context_tokens(explainer, sentence, target, word_agg="max", normalize=True, show=True):
#     """
#     Renders sentence with interactive token highlights.
#     - Blue  = token supports target prediction
#     - Red   = token opposes target prediction
#     - Gold  = the [MASK] position, labelled with the target word
#     Hovering any token shows its attribution score.
#     """
#     out = explainer.explain(
#         [sentence],
#         [[target]],
#         normalize=normalize,
#         return_word_scores=True,
#         word_agg=word_agg
#     )[0][target]

#     if out.get("skipped", False):
#         msg = f"Skipped: {out.get('reason', 'unknown reason')}"
#         if show:
#             display(HTML(f"<pre>{escape(msg)}</pre>"))
#         return msg

#     rows = out["word_attributions"]  # [(word, score), ...]
#     mask_tok = getattr(explainer.tokenizer, "mask_token", "[MASK]")
#     max_abs = max((abs(s) for w, s in rows if w != mask_tok), default=0.0)

#     container_id = f"tokviz_{uuid.uuid4().hex}"

#     token_spans = []
#     for word, score in rows:
#         if word == mask_tok:
#             # Gold pill showing [MASK] → target
#             span = (
#                 f"<span class='tok' data-score='(masked position)' "
#                 f"style='background:rgba(255,193,7,0.85); color:#000; font-weight:bold; "
#                 f"padding:2px 8px; margin:1px; border-radius:4px; cursor:default; "
#                 f"outline: 2px solid rgba(200,150,0,0.8);'>"
#                 f"[{escape(target)}]</span>"
#             )
#         else:
#             color = _attr_to_rgba(score, max_abs)
#             span = (
#                 f"<span class='tok' data-score='{score:.6f}' "
#                 f"style='background:{color}; padding:2px 4px; margin:1px; "
#                 f"border-radius:4px; cursor:default;'>"
#                 f"{escape(word)}</span>"
#             )
#         token_spans.append(span)

#     html = f"""
#     <div id="{container_id}">
#       <div style='margin-bottom:6px;'>
#         <b>Target:</b> <code>{escape(str(target))}</code>
#       </div>
#       <div style='margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;'>
#         <span style='background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;'>&#9646; predicts</span>
#         <span style='background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;'>&#9646; opposes</span>
#         <span style='background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;'>[target] mask position</span>
#       </div>
#       <div style='line-height:2.4; font-size:15px;'>
#         {' '.join(token_spans)}
#       </div>
#       <div class='tok-tooltip'
#            style='display:none; position:fixed; z-index:9999; pointer-events:none;
#                   background:#111; color:#fff; padding:5px 10px;
#                   border-radius:6px; font-size:12px; font-family:monospace;'>
#       </div>
#     </div>
#     <script>
#     (function() {{
#       const root = document.getElementById("{container_id}");
#       if (!root) return;
#       const tip  = root.querySelector(".tok-tooltip");
#       const toks = root.querySelectorAll(".tok");

#       toks.forEach(el => {{
#         el.addEventListener("mouseenter", () => {{
#           const raw = el.dataset.score;
#           const parsed = parseFloat(raw);
#           tip.textContent = isNaN(parsed)
#             ? raw
#             : "score = " + parsed.toFixed(4) + (parsed > 0 ? "  ▲ predicts" : "  ▼ opposes");
#           tip.style.display = "block";
#         }});
#         el.addEventListener("mousemove", (e) => {{
#           tip.style.left = (e.clientX + 14) + "px";
#           tip.style.top  = (e.clientY + 14) + "px";
#         }});
#         el.addEventListener("mouseleave", () => {{
#           tip.style.display = "none";
#         }});
#       }});
#     }})();
#     </script>
#     """

#     if show:
#         display(HTML(html))
#     return html

# def highlight_context_tokens_multi_target(explainer, sentence, targets, word_agg="max", normalize=True):
#     """
#     Renders one highlighted sentence per target.
#     Returns dict[target] -> html string.
#     """
#     rendered = {}
#     for t in targets:
#         rendered[t] = highlight_context_tokens(
#             explainer, sentence, t, word_agg=word_agg, normalize=normalize, show=True
#         )
#     return rendered
# ...existing code...

In [11]:

sentence = data_df.iloc[0]['maskedSentence']
highlight_context_tokens(explainer_new, sentence, target="slave", word_agg="max")


'\n    <div id="tokviz_40776be7813846b598ffc1474027df26">\n      <div style=\'margin-bottom:6px;\'>\n        <b>Target:</b> <code>slave</code>\n      </div>\n      <div style=\'margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;\'>\n        <span style=\'background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;\'>&#9646; predicts</span>\n        <span style=\'background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;\'>&#9646; opposes</span>\n        <span style=\'background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;\'>[target] mask position</span>\n      </div>\n      <div style=\'line-height:2.4; font-size:15px;\'>\n        <span class=\'tok\' data-score=\'0.037372\' style=\'background:rgba(30, 136, 229, 0.155); padding:2px 4px; margin:1px; border-radius:4px; cursor:default;\'>technical</span> <span class=\'tok\' data-score=\'0.038193\' style=\'background:rgba(30, 136, 229, 0.156); padding:2px 4px; 

In [ ]:


# def _iter_comparison_rows(comparison, target):
#     """
#     Yields (sent_idx, rows) where rows is list[(word, s1, s2, diff)].
#     Safely skips malformed / skipped entries.
#     """
#     for sent_idx, sent_comp in enumerate(comparison):
#         if not isinstance(sent_comp, dict) or target not in sent_comp:
#             continue

#         entry = sent_comp[target]

#         # Skip entries like {"skipped": True, ...}
#         if isinstance(entry, dict):
#             if entry.get("skipped", False):
#                 continue
#             continue

#         # Normal entries are list[(word, s1, s2, diff)]
#         if not isinstance(entry, list):
#             continue

#         rows = []
#         for item in entry:
#             if isinstance(item, (list, tuple)) and len(item) == 4:
#                 word, s1, s2, diff = item
#                 rows.append((word, float(s1), float(s2), float(diff)))

#         if rows:
#             yield sent_idx, rows


# def plot_model_comparison_bar(comparison, target, top_n=15):
#     stats = analyze_comparison(comparison, target, top_n=top_n)
#     if not stats:
#         print(f"No valid comparison rows found for target='{target}'.")
#         return

#     words = [s[0] for s in stats]
#     m1_scores = [s[1] for s in stats]
#     m2_scores = [s[2] for s in stats]

#     x = np.arange(len(words))
#     width = 0.35

#     fig, ax = plt.subplots(figsize=(14, 7))
#     ax.barh(x - width / 2, m1_scores, width, label="Model 1 (1760-1900)", alpha=0.8)
#     ax.barh(x + width / 2, m2_scores, width, label="Model 2 (1760-1850)", alpha=0.8)

#     ax.set_yticks(x)
#     ax.set_yticklabels(words)
#     ax.set_xlabel("Attribution Score", fontsize=12)
#     ax.set_title(f"Model Comparison: Top {top_n} Predictors for '{target}'", fontsize=14, fontweight="bold")
#     ax.axvline(x=0, color="black", linestyle="--", linewidth=0.8)
#     ax.legend(fontsize=11)
#     ax.grid(axis="x", alpha=0.3)

#     plt.tight_layout()
#     plt.show()



# def plot_scatter_model_comparison(comparison, target, top_n=25):
#     sent_data = []
#     for _, rows in _iter_comparison_rows(comparison, target):
#         for word, s1, s2, _ in rows:
#             sent_data.append((word, s1, s2))

#     if not sent_data:
#         print(f"No valid comparison rows found for target='{target}'.")
#         return

#     word_agg = {}
#     for word, s1, s2 in sent_data:
#         word_agg.setdefault(word, {"s1": [], "s2": []})
#         word_agg[word]["s1"].append(s1)
#         word_agg[word]["s2"].append(s2)

#     word_means = []
#     for word, vals in word_agg.items():
#         m1 = float(np.mean(vals["s1"]))
#         m2 = float(np.mean(vals["s2"]))
#         word_means.append((word, m1, m2, abs(m2 - m1)))

#     word_means.sort(key=lambda x: x[3], reverse=True)
#     word_means = word_means[:top_n]

#     words = [w[0] for w in word_means]
#     m1_vals = [w[1] for w in word_means]
#     m2_vals = [w[2] for w in word_means]
#     diffs = [w[3] for w in word_means]

#     fig, ax = plt.subplots(figsize=(10, 10))
#     scatter = ax.scatter(m1_vals, m2_vals, s=200, c=diffs, cmap="YlOrRd", alpha=0.6, edgecolors="black", linewidth=1)

#     for i, word in enumerate(words):
#         ax.annotate(word, (m1_vals[i], m2_vals[i]), fontsize=9, ha="center", va="center")

#     lim_min = min(min(m1_vals), min(m2_vals)) * 0.9
#     lim_max = max(max(m1_vals), max(m2_vals)) * 1.1
#     ax.plot([lim_min, lim_max], [lim_min, lim_max], "k--", alpha=0.3, linewidth=2, label="Equal scores")

#     ax.set_xlim(lim_min, lim_max)
#     ax.set_ylim(lim_min, lim_max)
#     ax.set_xlabel("Model 1 Attribution Score (1760-1900)", fontsize=12)
#     ax.set_ylabel("Model 2 Attribution Score (1760-1850)", fontsize=12)
#     ax.set_title(f"Model Attribution Comparison for '{target}'", fontsize=14, fontweight="bold")
#     ax.grid(alpha=0.3)

#     plt.colorbar(scatter, ax=ax, label="Absolute Difference")
#     ax.legend()
#     plt.tight_layout()
#     plt.show()



# def export_comparison_csv(comparison, target, output_file="model_comparison.csv"):
#     rows_out = []
#     for sent_idx, rows in _iter_comparison_rows(comparison, target):
#         for word, s1, s2, diff in rows:
#             rows_out.append({
#                 "sentence_idx": sent_idx,
#                 "word": word,
#                 "model1_score": float(s1),
#                 "model2_score": float(s2),
#                 "difference": float(diff),
#             })

#     comp_df = pd.DataFrame(rows_out)
#     comp_df.to_csv(output_file, index=False)
#     print(f"Comparison exported to {output_file} ({len(comp_df)} rows)")
#     return comp_df


In [19]:
# Cell 12: Generate comparison visualizations

# 1. Side-by-side bar chart
plot_model_comparison_bar(comparison, target="slave", top_n=15)

# 3. Scatter plot (correlation)
plot_scatter_model_comparison(comparison, target="slave", top_n=25)

NameError: name 'np' is not defined

In [ ]:
# ...existing code...
from html import escape
from IPython.display import HTML, display
import numpy as np
import uuid

# def _safe_sentence_rows(comparison, sent_idx, target):
#     """Return normalized rows: [(word, old_score, new_score, diff), ...] or None."""
#     if sent_idx < 0 or sent_idx >= len(comparison):
#         return None
#     sent_comp = comparison[sent_idx]
#     if not isinstance(sent_comp, dict) or target not in sent_comp:
#         return None

#     entry = sent_comp[target]
#     if isinstance(entry, dict):   # skipped/malformed record
#         return None
#     if not isinstance(entry, list):
#         return None

#     rows = []
#     for item in entry:
#         if isinstance(item, (list, tuple)) and len(item) == 4:
#             w, old_s, new_s, d = item
#             rows.append((str(w), float(old_s), float(new_s), float(d)))
#     return rows if rows else None

# def render_top_shift_sentences(
#     texts,
#     comparison,
#     target,
#     top_k=5,
#     score_mode="mean_abs",   # "mean_abs" or "max_abs"
#     show=True
# ):
#     """
#     Render top sentences where model change is largest for a given target.
#     comparison rows are expected as: (word, old_score, new_score, diff=old-new).
#     Blue = toward old model (diff > 0), Red = toward new model (diff < 0).
#     """
#     ranked = []
#     for i, sent in enumerate(texts):
#         rows = _safe_sentence_rows(comparison, i, target)
#         if not rows:
#             continue
#         diffs = np.array([r[3] for r in rows], dtype=float)
#         shift = float(np.mean(np.abs(diffs))) if score_mode == "mean_abs" else float(np.max(np.abs(diffs)))
#         ranked.append((i, sent, rows, shift))

#     ranked.sort(key=lambda x: x[3], reverse=True)
#     ranked = ranked[:top_k]

#     if not ranked:
#         msg = f"No valid rows found for target='{target}'."
#         if show:
#             print(msg)
#         return []

#     rendered = []
#     for rank, (sent_idx, sent, rows, shift) in enumerate(ranked, start=1):
#         container_id = f"shiftviz_{uuid.uuid4().hex}"

#         # normalize intensity by |diff|
#         max_abs = max(abs(d) for _, _, _, d in rows) if rows else 0.0
#         max_abs = max(max_abs, 1e-12)

#         token_spans = []
#         for w, old_s, new_s, d in rows:
#             if w == "[MASK]":
#                 span = (
#                     f"<span class='tok' data-tip='[MASK] position | target={escape(str(target))}' "
#                     f"style='background:rgba(255,193,7,0.90); color:#111; font-weight:bold; "
#                     f"padding:2px 8px; margin:1px; border-radius:4px; outline:2px solid rgba(200,150,0,0.9);'>"
#                     f"[{escape(str(target))}]</span>"
#                 )
#             else:
#                 strength = min(abs(d) / max_abs, 1.0)
#                 alpha = 0.12 + 0.78 * strength
#                 # diff = old - new; positive => old stronger (blue), negative => new stronger (red)
#                 bg = f"rgba(30,136,229,{alpha:.3f})" if d > 0 else f"rgba(229,57,53,{alpha:.3f})"
#                 tip = f"{escape(w)} | old={old_s:.4f} | new={new_s:.4f} | diff(old-new)={d:.4f}"
#                 span = (
#                     f"<span class='tok' data-tip='{tip}' "
#                     f"style='background:{bg}; padding:2px 4px; margin:1px; border-radius:4px; cursor:default;'>"
#                     f"{escape(w)}</span>"
#                 )
#             token_spans.append(span)

#         html = f"""
#         <div id="{container_id}" style="margin:10px 0 18px 0;">
#           <div style="margin-bottom:6px;">
#             <b>#{rank}</b> sentence_idx=<code>{sent_idx}</code> | shift=<code>{shift:.4f}</code>
#           </div>
#           <div style="margin-bottom:6px; color:#444;">
#             {escape(sent)}
#           </div>
#           <div style="margin:6px 0 10px 0; font-size:13px; display:flex; gap:8px; align-items:center;">
#             <span style="background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;">blue: toward old</span>
#             <span style="background:rgba(229,57,53,0.35); padding:2px 8px; border-radius:4px;">red: toward new</span>
#             <span style="background:rgba(255,193,7,0.90); padding:2px 8px; border-radius:4px; font-weight:bold;">[target] mask</span>
#           </div>
#           <div style="line-height:2.3; font-size:15px;">
#             {' '.join(token_spans)}
#           </div>
#           <div class="tok-tooltip"
#                style="display:none; position:fixed; z-index:9999; pointer-events:none;
#                       background:#111; color:#fff; padding:6px 10px; border-radius:6px;
#                       font-size:12px; font-family:monospace; max-width:70vw; white-space:nowrap; overflow:hidden; text-overflow:ellipsis;">
#           </div>
#         </div>
#         <script>
#         (function() {{
#           const root = document.getElementById("{container_id}");
#           if (!root) return;
#           const tip = root.querySelector(".tok-tooltip");
#           root.querySelectorAll(".tok").forEach(el => {{
#             el.addEventListener("mouseenter", () => {{
#               tip.textContent = el.dataset.tip || "";
#               tip.style.display = "block";
#             }});
#             el.addEventListener("mousemove", (e) => {{
#               tip.style.left = (e.clientX + 14) + "px";
#               tip.style.top  = (e.clientY + 14) + "px";
#             }});
#             el.addEventListener("mouseleave", () => {{
#               tip.style.display = "none";
#             }});
#           }});
#         }})();
#         </script>
#         """
#         rendered.append(html)
#         if show:
#             display(HTML(html))

#     return rendered
# ...existing code...

In [20]:
render_top_shift_sentences(
    texts=texts,
    comparison=comparison,
    target="slave",
    top_k=5,
    score_mode="mean_abs",
    show=True
)


NameError: name 'np' is not defined